# VisWord 05 — Per-row interpretability

Runs the interpret module on every trained row under `MyDrive/VISWORD/runs/`. Produces:

- `attention_sampleN.png`: last-block CLS attention heatmap (any ViT)
- `patch_triplet_N/patch_neighbours_sampleN.png`: patch-level NN maps
- `salad_clusters_sampleN.png` + `dustbin_map_sampleN.png`: SALAD-only
- `dustbin_evolution.png`: dustbin mass over training steps
- `cls_vs_vlad.png`: CLS-half vs VLAD-half cosine attribution

**Runtime:** T4 enough.  **Wallclock:** ~5 min per run × number of rows.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
from pathlib import Path
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['TORCH_HOME'] = f'{PROJECT}/hf_cache/torch'
sys.path.insert(0, f'{REPO_DIR}/src')
sys.path.insert(0, f'{REPO_DIR}/third_party/salad')
%cd $REPO_DIR

## Discover run dirs that have checkpoints

In [ ]:
runs_root = Path(PROJECT) / 'runs'
trained_runs = sorted([d for d in runs_root.iterdir() if (d / 'checkpoints' / 'best_phase1.pt').exists()])
print(f'{len(trained_runs)} trained runs found:')
for d in trained_runs: print(' ', d.name)

## Invoke `visword.interpret` on each

In [ ]:
for d in trained_runs:
    if (d / 'interpret' / 'attention_sample0.png').exists():
        print(f'skip {d.name} (already has interpret dir)')
        continue
    print(f'\n=== {d.name} ===')
    subprocess.run(
        ['python', '-u', '-m', 'visword.interpret', '--run-dir', str(d), '--k', '4'],
        check=False,
        env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'},
    )

## Display a few artefacts inline

In [ ]:
from IPython.display import Image, display, Markdown
for d in trained_runs:
    interp = d / 'interpret'
    if not interp.exists(): continue
    display(Markdown(f'### {d.name}'))
    for png in sorted(interp.glob('*.png'))[:6]:
        display(Image(str(png), width=400))